# LangGraph Pentest Agent

This notebook walks through wiring Wintermute's data models into a stateful
LangGraph agent. The shape we're building:

```
Ticket  ──>  read_ticket_node  ──>  planning_node  ──>  execution_node  ──>  reporting_node
                                       │                       │                  │
                                       ▼                       ▼                  ▼
                                  TestPlan +              LLM + tools         Vulnerability
                                  TestCaseRun                                  + run.status
```

Each node mutates a shared `AgentState` (`TypedDict`) so the graph carries enough
context across edges to drive a real engagement: pull a Jira/Bugzilla ticket, derive
scope, generate a `TestPlan`, run hardware tools through an LLM, and write
`Vulnerability` findings back into the live `Operation`.

> **Dependencies.** LangGraph is *not* a Wintermute dependency — install it locally
> with `pip install langgraph langchain-anthropic` to actually compile and run the
> graph. Cells that need it are clearly marked; the cells that exercise Wintermute's
> data models (which is most of the interesting pentest-state plumbing) run as-is.

See **Notebook 07 — Programmatic Hardware Cartridges** for the cartridge primitives
this agent calls.

## Imports & State Definition

We import:

- Wintermute's domain models (`Operation`, `TestPlan`, `TestCase`, `TestCaseRun`,
  `RunStatus`, `Vulnerability`) plus the `Ticket` system.
- The global tool registry (`tools`) so we can inspect what's available to bind to
  the LLM.
- LangChain message types and LangGraph's `StateGraph` — these may be unavailable
  depending on what's installed; the `import` is wrapped so the notebook still
  loads on a stock Wintermute env.

`AgentState` carries `messages` (the chat history), `current_ticket`, the active
`TestCaseRun`, and the running list of `findings` produced during execution.

In [ ]:
from __future__ import annotations

import io
from typing import Any, List, Optional, TypedDict

from rich.console import Console

from wintermute.ai.tools_runtime import tools as global_tool_registry
from wintermute.core import (
    Operation,
    RunStatus,
    TestCase,
    TestCaseRun,
    TestPlan,
)
from wintermute.findings import Vulnerability
from wintermute.tickets import InMemoryBackend, Ticket

# LangGraph + LangChain are optional — the data-model nodes work without them.
# We import only the symbols our compiled-graph cell actually references; the
# message types referenced inline in docstrings (AIMessage / HumanMessage /
# ToolMessage) are documented as commented production wiring further down.
try:
    from langgraph.graph import END, StateGraph  # type: ignore[import-not-found]

    HAS_LANGGRAPH = True
except ImportError:
    HAS_LANGGRAPH = False


class AgentState(TypedDict, total=False):
    """Shared state passed between graph nodes."""

    messages: List[Any]  # list[BaseMessage] when LangChain is installed
    current_ticket: Optional[Ticket]
    active_test_run: Optional[TestCaseRun]
    findings: List[Vulnerability]
    operation: Operation


print(f"LangGraph available: {HAS_LANGGRAPH}")
print(f"Tools currently registered: {len(global_tool_registry._tools)}")

## Bootstrapping the Operation + Ticket Backend

The agent reads tickets through Wintermute's `Ticket` metaclass, which delegates
static `Ticket.create()` / `Ticket.read()` to a registered backend (Jira, Bugzilla,
Salesforce, in-memory). For the demo we register `InMemoryBackend` once and create
one ticket inside it so `read_ticket_node` has something to pull.

In [ ]:
# Fresh operation + ticket backend per-notebook so re-runs stay clean.
operation = Operation(operation_name="acme-pentest-2026-Q2")
Ticket.register_backend("mem", InMemoryBackend(), make_default=True)

ticket_id = Ticket.create(
    title="I2C EEPROM extraction on MCIO",
    description=(
        "Analyze I2C EEPROM on MCIO port. Suspected hardcoded credentials "
        "in flash dump. Target: rasp1 (10.0.0.5). Bus: I2C-2, Address: 0x50."
    ),
    requester="alice@acme",
    assignee="red-team",
)
print(f"Created ticket {ticket_id}")

# Confirm round-trip read works.
_t = Ticket.read(ticket_id)
print(f"Title:       {_t.data.title}")
print(f"Status:      {_t.data.status}")
print(f"Description: {_t.data.description[:60]}...")

## Node 1 — `read_ticket_node`

Pulls the ticket through `Ticket.read()` and parses its description for scope hints.
Real engagements would feed this into an LLM for natural-language scope extraction;
we keep it deterministic so the rest of the graph remains testable.

In [ ]:
import re


def read_ticket_node(state: AgentState) -> AgentState:
    """Pull the ticket and seed the conversation with the parsed scope."""
    tid = state.get("current_ticket")  # might be a Ticket OR an id string
    if isinstance(tid, str):
        ticket = Ticket.read(tid)
    elif isinstance(tid, Ticket):
        ticket = tid
    else:
        raise ValueError("AgentState.current_ticket missing")

    desc = ticket.data.description
    scope = {
        "target_host": (
            re.search(r"Target:\s*(\S+)", desc).group(1)
            if re.search(r"Target:\s*(\S+)", desc)
            else "unknown"
        ),
        "bus": (
            re.search(r"Bus:\s*(\S+),?", desc).group(1)
            if re.search(r"Bus:\s*(\S+),?", desc)
            else "unknown"
        ),
        "address": (
            re.search(r"Address:\s*(0x[0-9a-fA-F]+)", desc).group(1)
            if re.search(r"Address:\s*(0x[0-9a-fA-F]+)", desc)
            else ""
        ),
    }
    state["current_ticket"] = ticket
    state.setdefault("messages", []).append(
        {
            "role": "system",
            "content": (
                f"Ticket {ticket.ticket_id}: {ticket.data.title}. Scope: {scope}"
            ),
        }
    )
    state.setdefault("findings", [])
    return state


# Drive the node manually (the way LangGraph would invoke it).
agent_state: AgentState = {
    "current_ticket": ticket_id,
    "operation": operation,
}
agent_state = read_ticket_node(agent_state)
print("Messages so far:")
for m in agent_state["messages"]:
    print(f"  [{m['role']}] {m['content']}")

## Node 2 — `planning_node`

Builds a `TestCase` for the scope, wraps it in a `TestPlan`, attaches the plan to
the operation, materialises the runs via `Operation.generateTestRuns()`, and marks
the freshly-created run as `IN_PROGRESS`. The agent's `active_test_run` slot points
at the live `TestCaseRun` so subsequent nodes can mutate it directly.

In [ ]:
def planning_node(state: AgentState) -> AgentState:
    """Synthesise a TestPlan / TestCaseRun from the ticket scope."""
    ticket = state["current_ticket"]
    if ticket is None:
        raise ValueError("planning_node requires a resolved ticket")
    op = state["operation"]

    test_case = TestCase(
        code=f"TC-{ticket.ticket_id}",
        name=f"Investigate: {ticket.data.title}",
        description=ticket.data.description,
    )
    plan = TestPlan(
        code=f"TP-{ticket.ticket_id}",
        name=f"Plan for {ticket.ticket_id}",
        description="Auto-generated by LangGraph agent",
        test_cases=[test_case],
    )
    op.addTestPlan(plan)
    op.generateTestRuns()

    # Find the run our planning step just created.
    run = next(r for r in op.test_runs if r.test_case_code == test_case.code)
    run.status = RunStatus.in_progress
    run.start()
    state["active_test_run"] = run
    return state


agent_state = planning_node(agent_state)
run = agent_state["active_test_run"]
assert run is not None
print(f"Active run: {run.run_id}")
print(f"  status:   {run.status.value}")
print(f"  started:  {run.started_at}")
print(f"Plans on operation: {[p.code for p in operation.test_plans]}")

## Node 3 — `execution_node`

This is where the LLM does the work. The pattern:

1. Fetch tool definitions from `global_tool_registry.get_definitions()` — the same
   schemas exposed via the MCP server and the in-console `tool_calling_chat` flow.
2. Bind them to a `ChatAnthropic` (or `ChatOpenAI`) instance via `.bind_tools(...)`.
3. Append a `HumanMessage` describing the scope, send the conversation through the
   model, and let it choose tool calls.
4. Execute each tool call against `global_tool_registry.call(...)` and feed the
   result back as a `ToolMessage` until the model emits a final answer.

Without an LLM the cell below substitutes a deterministic stub so the rest of the
graph can still be exercised — the production wiring is right there in the comment
block.

In [ ]:
# Make sure the firmware-analysis cartridge's methods are visible to the agent.
from wintermute.cartridges.manager import CartridgeManager

_manager = CartridgeManager()
if "firmware_analysis" not in _manager.list_loaded():
    _manager.load("firmware_analysis")
available = sorted(global_tool_registry._tools.keys())
print(f"Tools the agent can call: {available[:6]} ...")


def execution_node(state: AgentState) -> AgentState:
    """Run the LLM tool-calling loop.

    Production wiring (uncomment when langchain-anthropic is installed):

        from langchain_anthropic import ChatAnthropic
        from langchain_core.messages import HumanMessage, ToolMessage

        llm = ChatAnthropic(model="claude-sonnet-4-6")
        bound = llm.bind_tools(global_tool_registry.get_definitions())

        messages = state["messages"] + [HumanMessage(content="Investigate the scope.")]
        while True:
            ai = bound.invoke(messages)
            messages.append(ai)
            if not ai.tool_calls:
                break
            for call in ai.tool_calls:
                result = global_tool_registry.call(call["name"], call["args"])
                messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

    For this notebook we skip the LLM and call one tool deterministically so the
    state machine still demonstrably mutates `active_test_run`.
    """
    run = state["active_test_run"]
    if run is None:
        raise ValueError("execution_node requires an active test run")

    # Deterministic stand-in: pretend the LLM chose `extract_strings` on a
    # tiny synthetic blob and parsed the output. Real runs would loop through
    # multiple tool calls (entropy / strings / find_base_address / ...).
    import tempfile

    with tempfile.NamedTemporaryFile(
        suffix=".bin", delete=False, prefix="agent-fw-"
    ) as tmp:
        tmp.write(b"\x00" * 200 + b"admin:hunter2\x00secret-debug-key=ABCD\x00")
        blob_path = tmp.name

    raw = global_tool_registry.call(
        "extract_strings", {"file_path": blob_path, "min_length": 8}
    )
    # The registry wraps non-blob returns in `{"result": ...}`; an LLM
    # would see the same envelope and unwrap before reasoning over it.
    extract = raw.get("result", raw)
    state.setdefault("messages", []).append(
        {"role": "assistant", "content": f"extract_strings -> {extract}"}
    )
    state["_last_tool_result"] = extract  # type: ignore[typeddict-unknown-key]
    return state


agent_state = execution_node(agent_state)
interesting = agent_state["_last_tool_result"]["top_20_interesting_strings"]
print(f"Interesting strings the agent saw: {interesting}")

## Node 4 — `reporting_node`

Inspects the tool output, decides on a verdict, mutates the live `TestCaseRun`
(`status` + `finish()`), and appends a `Vulnerability` to `run.findings` for any
interesting result. Because we hand the agent the live `TestCaseRun` reference, the
mutation lands inside the operation's `test_runs` list — no separate write step
needed.

In [ ]:
def reporting_node(state: AgentState) -> AgentState:
    """Translate the LLM's tool output into a finding + run status update."""
    run = state["active_test_run"]
    if run is None:
        raise ValueError("reporting_node requires an active test run")

    last = state.get("_last_tool_result") or {}
    interesting = last.get("top_20_interesting_strings", [])
    creds_present = any(
        s for s in interesting if "admin" in s or "password" in s or "hunter2" in s
    )

    if creds_present:
        vuln = Vulnerability(
            title="Hardcoded credentials in I2C EEPROM",
            description=(
                f"Recovered admin string from EEPROM dump: "
                f"{interesting[0] if interesting else '(none)'}"
            ),
            cvss=8,
            threat="unauthorized device access via static credentials",
        )
        run.findings.append(vuln)
        state.setdefault("findings", []).append(vuln)
        run.status = RunStatus.failed  # vuln found → test "failed"
    else:
        run.status = RunStatus.passed
    run.finish()
    run.executed_by = "langgraph-agent"
    run.notes = (
        f"{run.notes}\nstrings_seen={len(interesting)} creds_present={creds_present}"
        if run.notes
        else f"strings_seen={len(interesting)} creds_present={creds_present}"
    )
    return state


agent_state = reporting_node(agent_state)
run = agent_state["active_test_run"]
assert run is not None
print(f"Run {run.run_id}")
print(f"  status:    {run.status.value}")
print(f"  ended_at:  {run.ended_at}")
print(f"  findings:  {[v.title for v in run.findings]}")
print(f"  notes:     {run.notes}")

## Compiling the LangGraph

When `langgraph` is available the four nodes are stitched into a `StateGraph`. The
graph below runs strictly linearly because each node refines the same state — but
it's trivial to add a conditional edge after `reporting_node` that loops back to
`execution_node` for follow-up tool calls.

In [ ]:
if HAS_LANGGRAPH:
    graph = StateGraph(AgentState)
    graph.add_node("read_ticket", read_ticket_node)
    graph.add_node("planning", planning_node)
    graph.add_node("execution", execution_node)
    graph.add_node("reporting", reporting_node)

    graph.set_entry_point("read_ticket")
    graph.add_edge("read_ticket", "planning")
    graph.add_edge("planning", "execution")
    graph.add_edge("execution", "reporting")
    graph.add_edge("reporting", END)
    compiled = graph.compile()

    # Stream events so the operator sees each transition land:
    #     for event in compiled.stream({...}, stream_mode="updates"):
    #         print(event)
    print("Graph compiled. Run with `compiled.invoke({...})` or `.stream(...)`.")
else:
    print("LangGraph not installed — graph compilation skipped.")
    print("The four nodes were already executed manually above and produced:")
    print(f"  active_test_run.status: {agent_state['active_test_run'].status.value}")
    print(f"  findings count:         {len(agent_state.get('findings', []))}")

## Final Operation Tree

Whether the graph ran via LangGraph or via the manual node calls above, the same
`Operation` instance accumulates: a `TestPlan`, its `TestCase`s, a `TestCaseRun` in
`failed` state, and the `Vulnerability` attached to that run. The console's
schema-aware tree renderer (`cmd_show`) walks the operation recursively via each
object's `__schema__` and prints every populated branch — proof that the agent
successfully populated the data models.

In [ ]:
from wintermute.WintermuteConsole import WintermuteConsole

# Mount a fresh console on this operation so cmd_show walks the agent's data.
console = WintermuteConsole()
console.operation = operation
console.rich_console = Console(file=io.StringIO(), force_terminal=False, width=200)
console.cmd_show()
buf = console.rich_console.file
assert isinstance(buf, io.StringIO)
print(buf.getvalue())

## Summary

Wiring Wintermute into a LangGraph agent is mostly a matter of letting each node
mutate live domain objects:

- **`read_ticket_node`** — `Ticket.read(...)` → seed scope into the conversation.
- **`planning_node`** — build a `TestPlan` from the scope, attach it to the
  operation, materialise `TestCaseRun`s, mark the active one `IN_PROGRESS`.
- **`execution_node`** — bind `global_tool_registry.get_definitions()` to a tool-
  calling LLM, loop until the model returns a final answer, accumulate tool
  outputs in the state.
- **`reporting_node`** — interpret the tool output, append `Vulnerability`s to
  `run.findings`, mark the run `passed` / `failed`, call `run.finish()`.

Because every step writes through to the `Operation` graph, the same data is
accessible to: the in-process REPL via `cmd_show`, external MCP clients via the
WintermuteMCP server, the `WorkspaceManager`'s blob descriptors, and any storage
backend registered with `Operation.register_backend(...)`. The agent doesn't need
its own persistence layer.